SETUP & INSTALLATION

In [ ]:

import os
import glob
import json
import textwrap
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = "/kaggle/input/fashion-product-images-small"

print("Environment ready!")
print("Dataset exists:", os.path.exists(DATA_DIR))


In [ ]:
import sys
import types


ftfy = types.ModuleType("ftfy")
ftfy.fix_text = lambda x: x

sys.modules["ftfy"] = ftfy

print("ftfy workaround added")

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if "bpe_simple_vocab_16e6" in file:
            print(os.path.join(root, file))

In [ ]:
import gzip
import shutil

source = "/kaggle/input/datasets/codenamekash/contrastive-language-image-pretraining-by-openai/clip/bpe_simple_vocab_16e6.txt"
destination = "/kaggle/working/bpe_simple_vocab_16e6.txt.gz"

with open(source, "rb") as f_in:
    with gzip.open(destination, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

print("Created:", destination)

In [ ]:
import shutil

shutil.copytree(
    "/kaggle/input/datasets/codenamekash/contrastive-language-image-pretraining-by-openai/clip",
    "/kaggle/working/clip",
    dirs_exist_ok=True
)

shutil.copy(
    "/kaggle/working/bpe_simple_vocab_16e6.txt.gz",
    "/kaggle/working/clip/bpe_simple_vocab_16e6.txt.gz"
)

print("CLIP folder ready")

In [ ]:
import sys
import os


sys.path = [p for p in sys.path if "codenamekash" not in p]


sys.path.insert(0, "/kaggle/working")


for m in list(sys.modules.keys()):
    if m.startswith("clip"):
        del sys.modules[m]

import clip

print("CLIP imported from:", clip.__file__)

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if "ViT_B_32" in file:
            print(os.path.join(root, file))

In [ ]:
import torch
import numpy as np
import torch.nn as nn
from clip.model import CLIP

device = "cuda" if torch.cuda.is_available() else "cpu"
model_path = "/kaggle/input/datasets/thedevastator/openaiclip-weights/ViT_B_32_vision_model.pt"

print("Loading local vision weights...")
vision_state_dict = torch.load(model_path, map_location="cpu")


original_init = CLIP.initialize_parameters
def dummy_initialize(self):
 
    if hasattr(self, 'visual') and hasattr(self.visual, 'initialize_parameters'):
        self.visual.initialize_parameters()
   
    if hasattr(self, 'text_projection') and self.text_projection is not None:
        nn.init.normal_(self.text_projection, std=self.transformer.width ** -0.5)


CLIP.initialize_parameters = dummy_initialize

print("Instantiating CLIP ViT-B/32 shell directly...")

model = CLIP(
    embed_dim=512,
    image_resolution=224,
    vision_layers=12,
    vision_width=768,
    vision_patch_size=32,
    context_length=77,
    vocab_size=49408,
    transformer_width=512,
    transformer_heads=8,
    transformer_layers=12
)


CLIP.initialize_parameters = original_init

print("Grafting trained vision weights...")

corrected_vision_dict = {}
for k, v in vision_state_dict.items():
    if not k.startswith("visual."):
        corrected_vision_dict[f"visual.{k}"] = v
    else:
        corrected_vision_dict[k] = v

model.load_state_dict(corrected_vision_dict, strict=False)

print("✓ Success! Vision model architecture is up and running offline.")
model = model.to(device)
model.eval()

In [ ]:
import os
import glob
import json
import textwrap
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import clip  # OpenAI CLIP
from clip.model import CLIP

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import normalize

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)



model_path = "/kaggle/input/datasets/thedevastator/openaiclip-weights/ViT_B_32_vision_model.pt"
vision_state_dict = torch.load(model_path, map_location="cpu")


original_init = CLIP.initialize_parameters
def dummy_initialize(self):
    if hasattr(self, 'visual') and hasattr(self.visual, 'initialize_parameters'):
        self.visual.initialize_parameters()
    if hasattr(self, 'text_projection') and self.text_projection is not None:
        nn.init.normal_(self.text_projection, std=self.transformer.width ** -0.5)

CLIP.initialize_parameters = dummy_initialize


model = CLIP(
    embed_dim=512, image_resolution=224, vision_layers=12, vision_width=768, 
    vision_patch_size=32, context_length=77, vocab_size=49408, 
    transformer_width=512, transformer_heads=8, transformer_layers=12
)

CLIP.initialize_parameters = original_init


corrected_vision_dict = {}
for k, v in vision_state_dict.items():
    if not k.startswith("visual."):
        corrected_vision_dict[f"visual.{k}"] = v
    else:
        corrected_vision_dict[k] = v

model.load_state_dict(corrected_vision_dict, strict=False)
model = model.to(device)
model.eval()


from clip.clip import _transform
preprocess = _transform(224)


print("CLIP model loaded successfully offline.")

In [ ]:

DATA_DIR = "/kaggle/input/datasets/paramaggarwal/fashion-product-images-small"
IMAGES_DIR = os.path.join(DATA_DIR, "images")
STYLES_CSV = os.path.join(DATA_DIR, "styles.csv")


df = pd.read_csv(STYLES_CSV, on_bad_lines="skip")
df = df.dropna(subset=["id", "productDisplayName"]).reset_index(drop=True)
df["id"] = df["id"].astype(int)


df["image_path"] = df["id"].apply(lambda x: os.path.join(IMAGES_DIR, f"{x}.jpg"))
df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)

print("Total usable products:", len(df))
df.head()

In [ ]:
SAMPLE_SIZE = 3000
if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print("Working set size:", len(df))

In [ ]:
@torch.no_grad()
def embed_images(image_paths, batch_size=64):
    all_feats = []
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i + batch_size]
        imgs = []
        for p in batch_paths:
            try:
                img = preprocess(Image.open(p).convert("RGB"))
            except Exception:
                img = torch.zeros(3, 224, 224)
            imgs.append(img)
        batch = torch.stack(imgs).to(device)
        feats = model.encode_image(batch)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        all_feats.append(feats.cpu().numpy())
        print(f"Embedded {min(i + batch_size, len(image_paths))}/{len(image_paths)}", end="\r")
    return np.vstack(all_feats)

image_embeddings = embed_images(df["image_path"].tolist())
print("\nImage embedding matrix shape:", image_embeddings.shape)

# Persist so we don't have to recompute
np.save("image_embeddings.npy", image_embeddings)
df.to_csv("products_indexed.csv", index=False)


TASK 1: Smart Product Recommendation Engine

In [ ]:

test_product_id = df["id"].iloc[0] 

print(f"Testing recommendations for Product ID: {test_product_id}")

recs_df = visualize_recommendations(test_product_id, df, image_embeddings, top_n=5)

In [ ]:
apparel_df = df[df["articleType"].str.lower().isin(["shirts", "tshirts", "jeans"])]

if not apparel_df.empty:
    test_clothing_id = apparel_df["id"].iloc[0]

    print(
        f"Testing clothing rules for Product ID: {test_clothing_id} "
        f"({df[df['id']==test_clothing_id]['articleType'].iloc[0]})"
    )

    recs_df = visualize_recommendations(
        test_clothing_id,
        df,
        image_embeddings,
        top_n=5
    )

    display(recs_df)

else:
    print("No shirts, tshirts, or jeans found in the sample.")

In [ ]:


COMPLEMENTARY_RULES = {
    "shoes": ["Socks", "Watches", "Bags", "Belts"],
    "sports shoes": ["Socks", "Watches", "Sports Accessories", "Bags"],
    "casual shoes": ["Socks", "Belts", "Watches", "Bags"],
    "formal shoes": ["Belts", "Watches", "Ties", "Cufflinks"],

    "tshirts": ["Jeans", "Shorts", "Casual Shoes", "Watches", "Caps"],
    "shirts": ["Trousers", "Jeans", "Belts", "Formal Shoes", "Watches", "Ties"],
    "jeans": ["Tshirts", "Shirts", "Casual Shoes", "Belts"],
    "trousers": ["Shirts", "Belts", "Formal Shoes"],

    "dresses": ["Heels", "Clutches", "Earrings", "Necklace and Chains"],

    "watches": ["Wallets", "Belts", "Sunglasses"],
    "sunglasses": ["Watches", "Caps", "Wallets"],

    "backpacks": ["Casual Shoes", "Tshirts", "Water Bottle"],
    "handbags": ["Heels", "Sunglasses", "Earrings"],

    "sports sandals": ["Socks", "Sports Accessories"],
    "flip flops": ["Shorts", "Tshirts"],
}


def get_complementary_categories(article_type, usage=None):
    """
    Returns complementary product categories based on article type.
    """

    key = str(article_type).strip().lower()

    if key in COMPLEMENTARY_RULES:
        return COMPLEMENTARY_RULES[key]

    # fallback
    return None


print("Complementary rules loaded successfully!")

print("\nShirts:")
print(get_complementary_categories("shirts"))

print("\nShoes:")
print(get_complementary_categories("shoes"))

print("\nUnknown category:")
print(get_complementary_categories("unknown"))

In [ ]:
def recommend_complementary(product_id, df, embeddings, top_n=5):
    """Return top_n complementary product recommendations for a given product id."""
    if product_id not in df["id"].values:
        raise ValueError(f"Product id {product_id} not found in dataset")

    row = df[df["id"] == product_id].iloc[0]
    idx = df.index[df["id"] == product_id][0]
    query_vec = embeddings[idx].reshape(1, -1)

    comp_categories = get_complementary_categories(row["articleType"], row["usage"])

    if comp_categories is not None:
        candidates = df[
            df["articleType"].isin(comp_categories) & (df["id"] != product_id)
        ]
    else:
       
        candidates = df[
            (df["usage"] == row["usage"])
            & (df["subCategory"] != row["subCategory"])
            & (df["id"] != product_id)
        ]

    if candidates.empty:
        return pd.DataFrame(columns=df.columns.tolist() + ["similarity"])

    cand_idx = candidates.index.to_numpy()
    cand_vecs = embeddings[cand_idx]
    sims = cosine_similarity(query_vec, cand_vecs)[0]

    candidates = candidates.copy()
    candidates["similarity"] = sims
    candidates = candidates.sort_values("similarity", ascending=False).head(top_n)

    return candidates[["id", "productDisplayName", "articleType", "baseColour", "similarity"]]


def visualize_recommendations(product_id, df, embeddings, top_n=5):
    row = df[df["id"] == product_id].iloc[0]
    recs = recommend_complementary(product_id, df, embeddings, top_n=top_n)

    n = len(recs) + 1
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.5))

    query_img = Image.open(row["image_path"]).convert("RGB")
    axes[0].imshow(query_img)
    axes[0].set_title(f"QUERY:\n{textwrap.fill(row['productDisplayName'], 20)}", fontsize=9, color="darkred")
    axes[0].axis("off")

    for ax, (_, rec) in zip(axes[1:], recs.iterrows()):
        img_path = df[df["id"] == rec["id"]]["image_path"].values[0]
        img = Image.open(img_path).convert("RGB")
        ax.imshow(img)
        ax.set_title(textwrap.fill(rec["productDisplayName"], 20) + f"\nsim={rec['similarity']:.2f}", fontsize=8)
        ax.axis("off")

    plt.suptitle("Complementary Product Recommendations", fontsize=13)
    plt.tight_layout()
    plt.show()
    return recs


In [ ]:

sample_row = df[df["articleType"].str.contains("Shoe", case=False, na=False)].iloc[0]
print("Query product:", sample_row["productDisplayName"], "| id:", sample_row["id"])

recs = visualize_recommendations(sample_row["id"], df, image_embeddings, top_n=5)
recs


In [ ]:

for pid in df.sample(3, random_state=1)["id"].tolist():
    print("="*80)
    visualize_recommendations(pid, df, image_embeddings, top_n=5)


TASK 2 Unique Product Catalog Creation

In [ ]:
from sklearn.metrics import pairwise_distances


cos_dist = pairwise_distances(image_embeddings, metric="cosine")

MIN_SAMPLES = 2

dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES, metric="precomputed")
cluster_labels = dbscan.fit_predict(cos_dist)

df["cluster_id"] = cluster_labels
n_dup_clusters = (df["cluster_id"] != -1).sum() > 0 and df[df["cluster_id"] != -1]["cluster_id"].nunique()
n_singletons = (df["cluster_id"] == -1).sum()

print(f"Found {n_dup_clusters} duplicate clusters (2+ near-identical products).")
print(f"Found {n_singletons} unique standalone products (no duplicates).")


In [ ]:
def build_unique_catalog(df, embeddings):
    catalog_rows = []
    duplicate_map = {} 
    singles = df[df["cluster_id"] == -1]
    for _, row in singles.iterrows():
        catalog_rows.append(row)
        duplicate_map[row["id"]] = [row["id"]]

   
    for cluster_id in sorted(df[df["cluster_id"] != -1]["cluster_id"].unique()):
        members = df[df["cluster_id"] == cluster_id]
        member_idx = members.index.to_numpy()
        member_vecs = embeddings[member_idx]
        centroid = member_vecs.mean(axis=0, keepdims=True)
        sims_to_centroid = cosine_similarity(centroid, member_vecs)[0]
        rep_pos = np.argmax(sims_to_centroid)
        rep_row = members.iloc[rep_pos]

        catalog_rows.append(rep_row)
        duplicate_map[rep_row["id"]] = members["id"].tolist()

    catalog_df = pd.DataFrame(catalog_rows).reset_index(drop=True)
    return catalog_df, duplicate_map


unique_catalog, duplicate_map = build_unique_catalog(df, image_embeddings)

print(f"Original catalog size:  {len(df)}")
print(f"Deduplicated catalog size: {len(unique_catalog)}")
print(f"Duplicates removed: {len(df) - len(unique_catalog)}")


In [ ]:

example_groups = {rep: members for rep, members in duplicate_map.items() if len(members) > 1}
print(f"Number of merged duplicate groups: {len(example_groups)}")

for rep_id, member_ids in list(example_groups.items())[:3]:
    print("\nRepresentative kept:", df[df['id'] == rep_id]['productDisplayName'].values[0], f"(id={rep_id})")
    print("Merged/removed duplicates:")
    for mid in member_ids:
        if mid != rep_id:
            print("   -", df[df['id'] == mid]['productDisplayName'].values[0], f"(id={mid})")


In [ ]:
def visualize_duplicate_group(rep_id, member_ids, df):
    imgs_ids = member_ids
    n = len(imgs_ids)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.5))
    if n == 1:
        axes = [axes]
    for ax, mid in zip(axes, imgs_ids):
        path = df[df["id"] == mid]["image_path"].values[0]
        name = df[df["id"] == mid]["productDisplayName"].values[0]
        img = Image.open(path).convert("RGB")
        ax.imshow(img)
        title = textwrap.fill(name, 18)
        if mid == rep_id:
            title = "[KEPT] " + title
        ax.set_title(title, fontsize=8, color=("darkgreen" if mid == rep_id else "black"))
        ax.axis("off")
    plt.suptitle("Duplicate Group -> Deduplicated to 1 Catalog Entry", fontsize=12)
    plt.tight_layout()
    plt.show()

# Visualize the largest duplicate group found
if example_groups:
    largest_rep = max(example_groups, key=lambda k: len(example_groups[k]))
    visualize_duplicate_group(largest_rep, example_groups[largest_rep], df)
else:
    print("No duplicate groups found in this sample; try increasing SAMPLE_SIZE or EPS.")


In [ ]:

unique_catalog_export = unique_catalog[["id", "productDisplayName", "articleType", "baseColour", "usage"]]
unique_catalog_export.to_csv("unique_product_catalog.csv", index=False)
unique_catalog_export.head(10)


TASK 3: Reverse Product Search (Text → Image)

In [ ]:
largest_rep = max(example_groups, key=lambda k: len(example_groups[k]))

sample_members = example_groups[largest_rep][:10]  
visualize_duplicate_group(
    largest_rep,
    sample_members,
    df
)

In [ ]:
largest_rep = max(example_groups, key=lambda k: len(example_groups[k]))

print("Representative ID:", largest_rep)
print("Number of duplicates:", len(example_groups[largest_rep]))

visualize_duplicate_group(
    largest_rep,
    example_groups[largest_rep],
    df
)

In [ ]:

results = visualize_text_search("blue casual shirt", df, image_embeddings, top_k=5)
results


In [ ]:

def search_interface():
    query = input("Enter a product search query (e.g. 'red summer dress'): ")
    if query.strip():
        visualize_text_search(query, df, image_embeddings, top_k=5)



for q in ["black leather handbag", "white sports shoes", "formal shirt for men"]:
    visualize_text_search(q, df, image_embeddings, top_k=5)
